# Agentic AI Agent — LangGraph + Groq + LLaMA
A ReAct-style agent built with **LangGraph** `StateGraph`. The graph has two nodes (`agent` and `tools`) connected in a cycle. The LLM decides when to call tools and when to stop.

**Graph topology:**
```
START → agent ──(has tool calls?)──► tools → agent → ...
                └──(no tool calls)──► END
```

**Tools available:**
- `calculator` — evaluates math expressions safely
- `get_datetime` — returns current date and time
- `search` — simulates a web search (stubbed)

> **Note:** Quiz questions and answers are at the end of this file.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'langgraph', 'langchain-groq', 'langchain-core', 'python-dotenv', '-q'])

In [ ]:
import os, math, datetime
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
)

In [ ]:
# ── Tools ─────────────────────────────────────────────────────────────────────

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports standard math operations and math.sqrt(), math.log(), etc."""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


@tool
def get_datetime() -> str:
    """Return the current date and time."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def search(query: str) -> str:
    """Search the web for a query and return a short summary."""
    # Stub — replace with Tavily or SerpAPI for live results
    return (
        f"[Search stub] No live search configured. "
        f"Plug in a real search API for results on: '{query}'"
    )


TOOLS = [calculator, get_datetime, search]

In [ ]:
# ── Build the LangGraph graph ──────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a helpful AI assistant with access to tools. "
    "Think step-by-step. Use tools whenever they give a more accurate answer. "
    "When you have enough information, give the user a clear final answer."
)

llm_with_tools = llm.bind_tools(TOOLS)


def agent_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


graph_builder = StateGraph(MessagesState)

graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", tools_condition)  # → tools or END
graph_builder.add_edge("tools", "agent")                       # cycle back

agent = graph_builder.compile()
print("Graph compiled successfully.")

In [ ]:
# ── Helper: run the agent and print step-by-step trace ────────────────────────

def run_agent(user_message: str):
    print(f"User: {user_message}\n")
    final_state = agent.invoke({"messages": [HumanMessage(content=user_message)]})

    for msg in final_state["messages"]:
        kind = type(msg).__name__
        if kind == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool call] {tc['name']}({tc['args']})")
        elif kind == "ToolMessage":
            print(f"  [Tool result] {msg.content}")

    answer = final_state["messages"][-1].content
    print(f"\nAgent: {answer}")
    return answer

In [ ]:
run_agent("What is the square root of 1764, and what day is today?")

In [ ]:
run_agent("If I invest $5000 at 7% annual interest compounded monthly for 10 years, how much will I have?")

In [ ]:
# Interactive — ask anything
user_q = input("Ask the agent: ")
print()
run_agent(user_q)

---

# Quiz: Agentic AI — Questions & Answers

---

### Q1. Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?

A stateful directed graph is a workflow where each node can access and update shared state as the agent executes. The graph consists of nodes connected by directed edges that define the flow of execution. Unlike a linear pipeline, it supports branching, conditional routing, loops, and revisiting previous steps based on intermediate results. A linear pipeline always follows a fixed sequence of operations without changing its execution path.

---

### Q2. Describe the role of nodes and edges in an agent workflow. Give an example of each.

Nodes represent individual operations or decision points in an agent workflow, such as invoking an LLM, calling an external API, or executing a tool. Edges define the transitions between nodes and determine the order of execution. For example, a "Search Tool" is a node, while the connection from the query analysis node to the search tool is an edge.

---

### Q3. What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.

Conditional routing directs incoming requests to different tools or workflows based on predefined conditions or the query's intent. It enables the agent to choose the most appropriate action.

**Example routing logic:**
- If the query contains mathematical expressions → route to the **Calculator Tool**.
- If the query requests a summary or keywords → route to the **Text Processing Tool**.
- Otherwise → route to the **General LLM Response** module.

---

### Q4. Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.

Cycles allow an agent to repeat a step until a condition is satisfied or a task succeeds. They are useful for handling temporary failures, iterative refinement, or repeated reasoning. A common example is retrying an API request after a timeout or network failure. Instead of terminating immediately, the agent retries the request a limited number of times before reporting an error.

---

### Q5. Explain how a single-agent system can simulate multi-agent behavior internally.

A single-agent system can mimic multiple agents by assigning different internal roles during execution. For example, it may first act as a planner, then as a tool selector, and finally as a response generator. Although only one agent is running, it performs multiple specialized functions sequentially, producing behavior similar to a multi-agent system with lower implementation complexity.

---

### Q6. What are JSON schema tools? How do they help in structuring tool inputs and outputs?

JSON Schema defines the expected structure, data types, and validation rules for data exchanged between an agent and external tools. It ensures that tool inputs contain all required fields in the correct format and that outputs follow a consistent structure. This reduces validation errors, simplifies parsing, and improves interoperability between different components.

---

### Q7. Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?

Sequential tool calls execute one after another, with each step depending on the output of the previous one. They are appropriate when tasks have dependencies. Parallel tool calls execute multiple independent tasks simultaneously, reducing overall latency. Parallel execution is preferred when tools do not rely on each other's results, while sequential execution is necessary when outputs from earlier steps are required for later ones.

---

### Q8. How would you implement error handling in a tool-using agent? Provide at least two strategies.

A robust agent should detect and recover from failures without terminating unnecessarily.
- **Exception handling:** Catch errors and return informative messages instead of crashing.
- **Retry logic:** Attempt the operation a limited number of times for temporary failures such as network issues.
- Additionally, maintaining logs of errors and tool responses helps diagnose failures and improve future reliability.

---

### Q9. What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?

Trajectory evaluation assesses the complete sequence of decisions, tool calls, and intermediate reasoning steps taken by an agent to solve a task. Unlike evaluating only the final output, it identifies inefficient routing, unnecessary tool usage, and reasoning errors that may still produce a correct answer. This makes it valuable for debugging, performance optimization, and improving overall agent behavior.

---

### Q10. Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?

**Task completion rate** is the percentage of tasks successfully completed according to predefined success criteria. **Cost metrics** measure the computational resources consumed, such as API usage, token consumption, execution time, or monetary cost. These metrics can be collected by monitoring agent executions across many tasks. They can be optimized by improving routing decisions, minimizing unnecessary tool invocations, reducing redundant LLM calls, and selecting more efficient models or algorithms while maintaining a high success rate.